# 09 - Build Territorial Graph

## Goal

This notebook builds the HERMES territorial graph.

The graph integrates:

- municipality attributes
- commuting flows

Each municipality becomes a graph node.

Each commuting flow becomes a directed edge weighted by the number of commuters.

The resulting graph is exported to GraphML for later analysis and simulation.

In [ ]:
# Imports

from hermes.config import (
    GRAPH_DIR,
)

from hermes.graph.builder import build_graph
from hermes.graph.validation import validate_graph
from hermes.graph.statistics import describe_graph
from hermes.graph.io import (export_graphml, load_graphml)

from hermes.loaders import (
    load_mobility,
    load_municipality,
)

import networkx as nx

In [ ]:
from hermes.data_catalog import get_dataset

dataset = get_dataset("municipality")

print(dataset.storage)
print(dataset.local_path)
print(dataset.local_path.exists())

In [ ]:
# Load datasets
municipality = load_municipality()
mobility = load_mobility()

In [ ]:
# ============================================================================
# Inspect destinations outside the territorial reference
# ============================================================================

municipality_codes = set(
    municipality["insee_code"].astype(str)
)

unknown_destinations = (
    mobility[
        ~mobility["destination_insee_code"].isin(
            municipality_codes
        )
    ][
        [
            "destination_insee_code",
            "destination_name",
        ]
    ]
    .drop_duplicates()
)

print(unknown_destinations.shape)

unknown_destinations

In [ ]:
unknown_destinations[
    "destination_insee_code"
].str[:2].value_counts()

In [ ]:
unknown_french_overseas = unknown_destinations[
    unknown_destinations["destination_insee_code"]
    .str.match(r"^(97|98)", na=False)
]

unknown_french_overseas

In [ ]:
print(unknown_french_overseas.shape)

unknown_french_overseas[
    "destination_insee_code"
].str[:3].value_counts()

In [ ]:
# Build graph

graph = build_graph(
    municipality,
    mobility,
)

In [ ]:
# Validate graph

validate_graph(graph)

In [ ]:
# Describe graph

describe_graph(graph)

In [ ]:
for node, attributes in graph.nodes(data=True):
    for key, value in attributes.items():
        if isinstance(value, bytes):
            print(node, key, type(value), len(value))
            break
    else:
        continue
    break

In [ ]:
# Save graph

export_graphml(
    graph,
    GRAPH_DIR / "municipality.graphml",
)

In [ ]:
# Final check

graph